+++
title = "DataFrame support"
menu = "main"
weight = 8
toc = true
aliases = ["dataframes", "polars"]
+++

Prince's PCA accepts pandas DataFrames, Polars DataFrames, and NumPy arrays. [Narwhals](https://narwhals-dev.github.io/narwhals/) provides the DataFrame compatibility layer, while Prince uses NumPy for the numerical computation.

Polars is optional. Install it alongside Prince if you want to use Polars DataFrames:

```sh
pip install prince polars
```

## Polars input

Pass a Polars DataFrame directly to `fit`, `transform`, or `fit_transform`.

In [1]:
from __future__ import annotations

import polars as pl

import prince

X = pl.DataFrame(
    {
        "height": [1.61, 1.72, 1.80, 1.68, 1.75],
        "weight": [54.0, 68.0, 81.0, 63.0, 75.0],
        "age": [23.0, 35.0, 29.0, 41.0, 32.0],
        "score": [7.2, 8.1, 8.8, 7.7, 8.4],
    }
)
X

height,weight,age,score
f64,f64,f64,f64
1.61,54.0,23.0,7.2
1.72,68.0,35.0,8.1
1.8,81.0,29.0,8.8
1.68,63.0,41.0,7.7
1.75,75.0,32.0,8.4


Prince returns a pandas DataFrame by default. This preserves the output behavior of earlier Prince versions. Because Polars and NumPy do not have an index, their default result uses a pandas `RangeIndex`. A pandas input keeps its original index.

In [2]:
coordinates = prince.PCA(n_components=2, random_state=42).fit_transform(X)
print(type(coordinates))
coordinates

<class 'pandas.DataFrame'>


component,0,1
0,2.850243,-1.114482
1,-0.190770,0.480283
2,-2.278634,-0.825293
3,0.743605,1.621341
4,-1.124443,-0.161849


## Polars output

Use scikit-learn's [`set_output`](https://scikit-learn.org/stable/auto_examples/miscellaneous/plot_set_output.html) API when the result should also be a Polars DataFrame.

In [3]:
pca = prince.PCA(n_components=2, random_state=42).set_output(transform="polars")
coordinates_polars = pca.fit_transform(X)
print(type(coordinates_polars))
coordinates_polars

<class 'polars.dataframe.frame.DataFrame'>


0,1
f64,f64
2.850243,-1.114482
-0.19077,0.480283
-2.278634,-0.825293
0.743605,1.621341
-1.124443,-0.161849


## Supplementary columns

Named supplementary columns work with both pandas and Polars inputs. Prince reports their column coordinates, but does not use them to fit the principal components.

In [4]:
pca = prince.PCA(n_components=2, random_state=42)
pca.fit(X, supplementary_columns=["score"])
print(pca.feature_names_in_)
pca.column_coordinates_

['height', 'weight', 'age']


component,0,1
variable,,
height,-0.990509,-0.131192
weight,-0.982043,-0.184239
age,-0.329233,0.944246
score,-0.981241,-0.189012


## NumPy input

PCA also accepts a two-dimensional NumPy array. Column names and supplementary columns are unavailable for array input, so columns are selected by position and the default result uses a `RangeIndex`.

## Current scope

This multi-DataFrame support currently applies to `prince.PCA`. The other Prince estimators still require pandas DataFrames. The Narwhals layer gives us a common path for extending Polars support to those estimators without maintaining separate implementations for each DataFrame library.